# Modelos de Regressão e Classificação Linear Aplicados a Dados de PIB e Eletromiografia

**Aluno(a)/equipe:** PREENCHER  
**Matrícula(s):** PREENCHER  
**Disciplina:** Inteligência Artificial Computacional  
**Universidade de Fortaleza - UNIFOR**

## Resumo

Neste trabalho foram implementados modelos de regressão e classificação baseados no método dos mínimos quadrados. A primeira parte usa o ano para estimar o PIB da China. A segunda usa sinais de dois músculos faciais para identificar cinco expressões. Foram comparados o MQO tradicional, sua versão regularizada e um modelo polinomial. A avaliação foi feita com 500 divisões aleatórias, usando 80% dos dados para treino e 20% para teste. O modelo polinomial apresentou o melhor resultado nas duas tarefas: $R^2$ médio de 0,9949 na regressão e acurácia média de 99,28% na classificação.

## 1. Introdução

A regressão e a classificação são problemas supervisionados, mas possuem objetivos diferentes. Na regressão, o resultado previsto é uma quantidade contínua. Na classificação, a saída indica a qual grupo uma amostra pertence. Mesmo assim, os dois problemas podem ser resolvidos com uma base matemática parecida quando a classificação é escrita como um modelo de múltiplas saídas.

Os algoritmos deste trabalho foram construídos com operações matriciais do NumPy, seguindo os códigos vistos em sala. Não foram usadas implementações prontas de bibliotecas de aprendizado de máquina. Além de comparar os modelos, buscamos observar se o desempenho continuava estável quando a divisão entre treino e teste era alterada.

In [ ]:
from pathlib import Path
import csv
import numpy as np

# Localiza a pasta do projeto mesmo se o notebook for aberto em outro diretório.
possiveis_raizes = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PASTA_PROJETO = next(
    pasta / 'trabalho_av1'
    for pasta in possiveis_raizes
    if (pasta / 'trabalho_av1').exists()
)
PASTA_DADOS = PASTA_PROJETO.parent / 'data set'
PASTA_RESULTADOS = PASTA_PROJETO / 'resultados'
print(PASTA_PROJETO)

## 2. Metodologia

Antes do ajuste, as características foram padronizadas por z-score: $z=(x-\mu)/\sigma$. A média e o desvio-padrão foram calculados apenas com o conjunto de treino. Essa etapa é importante porque o ano e os valores dos sensores possuem escalas altas e, no modelo polinomial, ainda são elevados a diferentes potências.

Com o intercepto, o modelo é escrito como $\hat{Y}=\Phi\beta$. No MQO tradicional, os parâmetros são calculados por

$$\beta=(\Phi^T\Phi)^{\dagger}\Phi^TY,$$

onde $\dagger$ representa a pseudoinversa. Para o modelo regularizado de Tikhonov, foi acrescentado o termo $\lambda I$:

$$\beta=(\Phi^T\Phi+\lambda I)^{\dagger}\Phi^TY.$$

A expansão polinomial usada foi $\Phi=[1,X,X^2,\ldots,X^q]$. Como nos códigos da aula, cada característica foi elevada separadamente, sem termos cruzados.

In [ ]:
china = np.loadtxt(PASTA_DADOS / 'china_gdp.csv', delimiter=',', skiprows=1)
emg = np.loadtxt(PASTA_DADOS / 'EMG1.csv')

X_reg, y_reg = china[:, :1], china[:, 1:]
X_cla, rotulos = emg[:, :2], emg[:, 2].astype(int)

print('Regressão:', X_reg.shape, y_reg.shape)
print('Classificação:', X_cla.shape, rotulos.shape)
print('Amostras por classe:', dict(zip(*np.unique(rotulos, return_counts=True))))

Na classificação, os rótulos foram convertidos em uma matriz com cinco colunas. A classe correta recebeu $+1$ e as demais receberam $-1$, repetindo a estratégia do exemplo de sala. O modelo gera cinco escores para cada amostra e a previsão corresponde à posição do maior escore, obtida com `argmax`.

A validação foi repetida 500 vezes. Em cada rodada, os dados foram embaralhados e separados em 80% para treino e 20% para teste. Foram armazenados MSE e $R^2$ na regressão e acurácia na classificação. Depois calculamos média, desvio-padrão, maior e menor valor. A semente aleatória foi fixada em 42 para permitir a reprodução do experimento.

## 3. Resultados da regressão

O conjunto `china_gdp.csv` possui 55 observações, entre 1960 e 2014. O gráfico deixa claro que o crescimento não segue uma reta: a mudança é pequena no início e se torna muito mais rápida nos anos finais. Por isso, já era esperado que o modelo linear tivesse dificuldade.

![Comparação inicial dos modelos](../trabalho_av1/resultados/regressao/03_modelos_iniciais.png)

Foram testadas ordens polinomiais de 1 a 10. A ordem 9 apresentou o melhor $R^2$ médio na avaliação preliminar, igual a 0,9965. A ordem 10 não melhorou o resultado, então o termo adicional não foi mantido.

### Resumo das 500 rodadas

| Modelo | MSE médio | $R^2$ médio | Desvio de $R^2$ |
|---|---:|---:|---:|
| Polinomial $q=9$ | **1,4544 × 10²²** | **0,9949** | **0,0145** |
| MQO tradicional | 3,3781 × 10²⁴ | -1,3214 | 9,5537 |
| Regularizado 0,25 | 3,3763 × 10²⁴ | -1,2942 | 9,4380 |
| Regularizado 0,50 | 3,3750 × 10²⁴ | -1,2676 | 9,3244 |
| Regularizado 0,75 | 3,3739 × 10²⁴ | -1,2415 | 9,2128 |
| Regularizado 1,00 | 3,3732 × 10²⁴ | -1,2160 | 9,1031 |

O polinômio reduziu bastante o erro e manteve $R^2$ próximo de 1. Já os modelos lineares tiveram $R^2$ médio negativo. Isso não indica um problema na métrica: significa que, em várias divisões, a reta foi pior do que uma previsão constante baseada na média. A regularização trouxe uma melhora pequena, mas não mudou a forma da curva. Nesse caso, o problema não era apenas o tamanho dos coeficientes; era tentar representar um crescimento curvo com uma reta.

## 4. Resultados da classificação

O conjunto de eletromiografia possui 50.000 amostras, duas características e cinco classes balanceadas, com 10.000 registros por expressão. O gráfico de dispersão mostra grupos com formatos bem diferentes. Algumas classes ficam próximas aos eixos, enquanto outras formam nuvens alongadas. Uma única fronteira reta não consegue acompanhar toda essa geometria.

![Distribuição das classes](../trabalho_av1/resultados/classificacao/02_dispersao_emg.png)

As ordens de 1 a 6 produziram acurácias preliminares de 72,53%, 94,49%, 97,64%, 99,24%, 99,26% e 99,30%. Escolhemos $q=4$ porque as ordens 5 e 6 melhoraram menos de 0,1 ponto percentual e exigiram mais operações.

### Resumo das 500 rodadas

| Modelo | Acurácia média | Desvio-padrão | Maior | Menor |
|---|---:|---:|---:|---:|
| MQO tradicional | 72,3888% | 0,6430% | 74,18% | 70,13% |
| Regularizado 0,25 | 72,3890% | 0,6428% | 74,18% | 70,13% |
| Polinomial $q=4$ | **99,2799%** | **0,0829%** | **99,48%** | **99,06%** |

![Distribuição das acurácias](../trabalho_av1/resultados/classificacao/06_distribuicao_acuracia.png)

O resultado do polinômio foi alto e também estável: mesmo a menor acurácia entre as 500 rodadas ficou acima de 99%. O MQO tradicional e o regularizado ficaram praticamente empatados. Com muitas amostras de treino, $\lambda=0,25$ representa uma penalidade pequena. Além disso, a regularização continua produzindo fronteiras retas. A melhora principal veio das potências até a quarta ordem, que permitiram desenhar regiões de decisão curvas.

## 5. Conclusões

Os experimentos mostraram que a forma do modelo foi mais importante do que a regularização nos dois problemas. No conjunto do PIB, o crescimento acelerado não pôde ser descrito adequadamente por uma reta. Nos sinais de eletromiografia, as classes também exigiram fronteiras curvas. Por isso, os modelos polinomiais tiveram uma diferença grande em relação às versões lineares.

A repetição das divisões de treino e teste ajudou a confirmar que o resultado não dependia de uma única escolha de amostras. Tanto o polinômio de ordem 9 na regressão quanto o de ordem 4 na classificação apresentaram desempenho alto e pouca variação. Também foi possível observar que aumentar a ordem além do necessário trouxe pouco ganho, reforçando a importância de comparar desempenho e complexidade antes de escolher o modelo final.

## Observação sobre o enunciado

O PDF não informa qual valor de $\lambda$ deve ser usado na classificação. Neste relatório foi adotado $\lambda=0,25$, o primeiro valor regularizado solicitado na parte de regressão. O enunciado também menciona modelos gaussianos bayesianos apenas no item sobre dimensões das matrizes, mas esses modelos não aparecem na lista de implementações nem na tabela final. Por esse motivo, a comparação foi feita com os três classificadores explicitamente pedidos.

## Referências

1. C. M. Bishop. *Pattern Recognition and Machine Learning*. Springer, 2006.
2. G. James, D. Witten, T. Hastie e R. Tibshirani. *An Introduction to Statistical Learning*. 2ª edição, Springer, 2021.
3. P. C. S. Barbosa. *Fundamentos da regressão e classificação linear*. Material da disciplina de Inteligência Artificial Computacional, Universidade de Fortaleza.